In [5]:
import os
import argparse
import sys
import pandas as pd
from plotnine import *

In [2]:
PREFIX = 'WGS-ONT-A0003-3616158-D26526'

In [ ]:
# working directory
WD = f'/lab_nas_home_user/Mark/ONT/{PREFIX}'
os.makedirs(WD, exist_ok=True)

# Set threads
CORES = int(round(os.cpu_count() * 0.8))

# Softwares
SAMTOOLS = '/Everythings/misc/samtools-1.21/samtools'
MAKEBLASTDB = "/Everythings/misc/ncbi-blast-2.13.0+/bin/makeblastdb"
BLASTN = "/Everythings/misc/ncbi-blast-2.13.0+/bin/blastn"
BAM2CPG = '/Everythings/misc/pb-CpG-tools-v2.3.2-x86_64-unknown-linux-gnu/bin/aligned_bam_to_cpg_scores'

# Path
TMP = '/raid/tmp/FSHD'
os.makedirs(TMP, exist_ok=True)
REF = '/nanopore_nas/pipeline/data/GCA_000001405.15_GRCh38_full_plus_hs38d1_analysis_set_modified.fasta'
QUERY = "/lab_nas_home_user/Mark/FSHD/querySeq"

# Other infos
blastTitle = ['QuerySeq-id', 'Seq-id', 'Identity%', 'Alignment-length', 'Mismatches', 'Gap', 'Q.Start', 'Q.End', 'S.Start', 'S.End', 'E-value', 'Bit-score']



### Merge BAM files ###
BAMS = f'/nanopore_nas/nanopore/{PREFIX}'
os.system(f'find {BAMS} -type f -name "*.bam" > {TMP}/bams.txt')
os.system(f'{SAMTOOLS} merge -b {TMP}/bams.txt -R chr4:190064502-190175845 -o - -@ {CORES} | {SAMTOOLS} sort -m 10G -u -@ {CORES} | {SAMTOOLS} view -b -@ {CORES} -o {TMP}/tmp.bam')
os.system(f'{SAMTOOLS} index -@ {CORES} {TMP}/tmp.bam')

### Create fasta file from BAM file
os.system(f'{SAMTOOLS} fasta -@ {CORES} {TMP}/tmp.bam > {TMP}/tmp_chr4.fasta')

### Create BLAST database
DB = f'{TMP}/blastdb'
os.system(f'{MAKEBLASTDB} -in {TMP}/tmp_chr4.fasta -dbtype nucl -out {DB} -title {PREFIX}')

[M::bam2fq_mainloop] discarded 0 singletons
[M::bam2fq_mainloop] processed 397 reads




Building a new DB, current time: 03/24/2025 15:30:47
New DB name:   /raid/tmp/FSHD/blastdb
New DB title:  WGS-ONT-A0003-3616158-D26526
Sequence type: Nucleotide
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 397 sequences in 0.0468159 seconds.




0

In [8]:
### Sequence ID with head sequence (DBET)
os.system(f'{BLASTN} -db {DB} -query {QUERY}/DBET.fasta -out {TMP}/head.blast -num_threads {CORES} -outfmt 6')

head = (
    pd.read_table(f'{TMP}/head.blast', header=None, sep='\t', names=blastTitle)
        .drop(columns=['QuerySeq-id'])
        .sort_values(by='Identity%', ascending=False)
)
head

,Seq-id,Identity%,Alignment-length,Mismatches,Gap,Q.Start,Q.End,S.Start,S.End,E-value,Bit-score
59,ef8afac7-26e5-4504-9155-44fad1bb3bb7,100.000,133,0,0,3175,3307,20067,20199,8.680000e-65,246.0
320,158410e1-9f71-4ec7-8ddb-7dd7d1d6ab63,100.000,110,0,0,728,837,19062,19171,5.300000e-52,204.0
28,1c7befa5-06f0-457e-8a5e-19dc0e6bddbc,100.000,133,0,0,3175,3307,11202,11334,8.680000e-65,246.0
29,1c7befa5-06f0-457e-8a5e-19dc0e6bddbc,100.000,133,0,0,3175,3307,14479,14611,8.680000e-65,246.0
33,1c7befa5-06f0-457e-8a5e-19dc0e6bddbc,100.000,133,0,0,728,860,31767,31899,8.680000e-65,246.0
...,...,...,...,...,...,...,...,...,...,...,...
879,536275b6-85d1-4efb-abc8-2fed8b6d62c8,81.435,878,71,77,2217,3057,8729,9551,0.000000e+00,634.0
881,536275b6-85d1-4efb-abc8-2fed8b6d62c8,80.976,799,76,64,2275,3041,5576,6330,2.110000e-160,564.0
927,d9a070f1-9f2f-41cf-bf75-7ac07bacd7ff,79.375,160,10,15,1466,1620,1413,1272,4.310000e-18,91.6
1090,ba9de350-5d08-4f8f-90b8-6a3d303c807c,75.687,691,91,57,742,1411,4389,5023,3.980000e-73,274.0
